# K-IFRS Agent 테스트

Think Tool + K-IFRS 검색 도구 5종을 활용한 에이전트 테스트 노트북.

**사전 준비:**
- `.env`에 `ANTHROPIC_API_KEY` 설정
- `.env`에 `DATABASE_DIR=/path/to/_database` 설정 (K-IFRS 검색용)
- `.env`에 `UPSTAGE_API_KEY` 설정 (임베딩용)

**사용 도구:**
| 태그 | 도구 | 설명 |
|------|------|------|
| reasoning | `think_tool` | 전략적 사고/분석 |
| rag, kifrs | `kifrs_search` | 하이브리드 검색 + Reranker |
| rag, kifrs | `kifrs_fetch_paragraph` | 특정 문단 직접 조회 |
| rag, kifrs | `kifrs_find_referencing` | 역방향 참조 검색 |
| rag, kifrs | `kifrs_explore_related` | 기준서 참조 그래프 탐색 |
| rag, kifrs | `kifrs_term_definitions` | 용어정의(Appendix A) 조회 |

---
## 1. 환경 설정

In [1]:
import sys, os

# 패키지 상위 디렉토리를 경로에 추가
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

In [2]:
from Agent_Structure.config import settings

print(f"Provider:     {settings.default_provider}")
print(f"Model:        {settings.default_model}")
print(f"DATABASE_DIR: {settings.database_dir}")

# 필수 설정 확인
missing = settings.validate()
if missing:
    print(f"\n!! 누락된 키: {missing}")
if not settings.database_dir:
    print("\n!! DATABASE_DIR 환경변수가 설정되지 않았습니다.")
    print("   .env 파일에 DATABASE_DIR=/path/to/_database 를 추가하세요.")
else:
    print("\n* 모든 필수 설정이 완료되었습니다.")

Provider:     anthropic
Model:        claude-sonnet-4-5-20250929
DATABASE_DIR: /home/shin/Study/_database

* 모든 필수 설정이 완료되었습니다.


---
## 2. 도구 등록 확인

In [3]:
from Agent_Structure.tools import tool_registry

# 전체 도구 요약
print(tool_registry.summary())

=== ToolRegistry (7 tools) ===
  [search, web] web_search: Tavily API를 사용한 웹 검색.

    Args:
        query: 검색 쿼리
      
  [reasoning] think_tool: Tool for strategic reflection on research progress and decis
  [rag, kifrs] kifrs_search: K-IFRS(한국채택국제회계기준) 기준서에서 관련 내용을 검색합니다.
    BM25와 Dense Vecto
  [rag, kifrs] kifrs_fetch_paragraph: K-IFRS 기준서의 특정 문단 내용을 직접 조회합니다.
    교차참조('문단 XX 참조' 등)가 있을 때
  [rag, kifrs] kifrs_find_referencing: 특정 기준서를 참조하는 다른 기준서의 문단을 역방향 검색합니다.
    예: '제1109호를 다른 기준서가 
  [rag, kifrs] kifrs_explore_related: 기준서 참조 그래프를 탐색하여 관련 기준서를 발견합니다.
    특정 기준서와 직접·간접적으로 연결된 기준서
  [rag, kifrs] kifrs_term_definitions: K-IFRS 기준서의 용어정의(Appendix A)를 조회합니다.
    '금융자산', '리스', '수행의무


In [4]:
# 태그별 필터링 확인
kifrs_tools = tool_registry.get_by_tag("kifrs")
reasoning_tools = tool_registry.get_by_tag("reasoning")

print(f"[kifrs] 도구 {len(kifrs_tools)}개:")
for t in kifrs_tools:
    print(f"  - {t.__name__}")

print(f"\n[reasoning] 도구 {len(reasoning_tools)}개:")
for t in reasoning_tools:
    print(f"  - {t.__name__}")

[kifrs] 도구 5개:
  - kifrs_search
  - kifrs_fetch_paragraph
  - kifrs_find_referencing
  - kifrs_explore_related
  - kifrs_term_definitions

[reasoning] 도구 1개:
  - think_tool


---
## 3. 에이전트 생성

`exclude_tools`로 `web_search`를 제외하여 think + kifrs 도구만 사용하는 에이전트를 만듭니다.

In [5]:
from Agent_Structure.run_notebook import create_agent, run, stream

agent = create_agent(
    exclude_tools=["web_search", "example_tool"],
    system_prompt=(
        "당신은 K-IFRS(한국채택국제회계기준) 전문 AI 어시스턴트입니다. "
        "회계 기준서 검색 도구를 활용하여 정확한 근거에 기반한 답변을 제공합니다. "
        "복잡한 질문에는 think_tool로 분석 전략을 수립한 뒤 검색하세요."
    ),
)
print(f"에이전트 타입: {type(agent).__name__}")
print("에이전트가 준비되었습니다!")

에이전트 타입: CompiledStateGraph
에이전트가 준비되었습니다!


---
## 4. 기본 검색 테스트

`kifrs_search` 도구를 호출하는 단순 질문.

In [6]:
result = run(agent, "K-IFRS 제1115호의 수익인식 5단계를 검색해서 설명해주세요.", thread_id="test-1")
print(result)

K-IFRS 제1115호 '고객과의 계약에서 생기는 수익'의 **수익인식 5단계**는 다음과 같습니다:

## 수익인식 5단계 모형

### **(1) 1단계: 고객과의 계약 식별**
계약은 둘 이상의 당사자 사이에 집행 가능한 권리와 의무가 생기게 하는 합의입니다. K-IFRS 1115호의 요구사항은 정해진 조건을 충족하는 고객과 체결한 계약에만 적용됩니다. 경우에 따라 여러 계약을 결합하여 하나의 계약으로 회계처리하기도 합니다.

### **(2) 2단계: 수행의무 식별**
하나의 계약은 고객에게 재화나 용역을 이전하는 여러 약속을 포함할 수 있습니다. 그 재화나 용역들이 **구별된다면** 약속은 수행의무이고 별도로 회계처리합니다. 재화나 용역이 구별되려면:
- 고객이 재화나 용역 그 자체에서나 쉽게 구할 수 있는 다른 자원과 함께하여 효익을 얻을 수 있고,
- 그 약속을 계약 내의 다른 약속과 별도로 식별해 낼 수 있어야 합니다.

### **(3) 3단계: 거래가격 산정**
거래가격은 고객에게 약속한 재화나 용역을 이전하고 그 대가로 기업이 받을 권리를 갖게 될 것으로 예상하는 금액입니다. 거래가격 산정 시 고려사항:
- **변동대가**: 불확실성이 나중에 해소될 때 인식된 누적 수익 금액 중 유의적인 부분을 되돌리지 않을 가능성이 매우 높은 정도까지만 포함
- **유의적인 금융요소**: 계약에 포함된 경우 화폐의 시간가치 영향을 조정
- **비현금 대가**: 현금 외의 형태로 지급되는 대가
- **고객에게 지급하는 대가**: 거래가격에서 조정

### **(4) 4단계: 거래가격을 계약 내 수행의무에 배분**
거래가격은 일반적으로 계약에서 약속한 **각 구별되는 재화나 용역의 상대적 개별 판매가격을 기준**으로 배분합니다. 개별 판매가격을 관측할 수 없다면 추정해야 합니다. 경우에 따라 할인액이나 변동대가를 일부 수행의무에만 배분할 수 있습니다.

### **(5) 5단계: 수행의무를 이행할 때(또는 기간에 걸쳐 이행하는 대로) 수익 인식**
기업이 

---
## 5. 교차참조 테스트

`kifrs_fetch_paragraph`로 특정 문단을 직접 조회합니다.

In [ ]:
result = run(agent, "K-IFRS 제1109호 문단 5.5.1의 내용을 찾아서 보여주세요.", thread_id="test-2")
print(result)

---
## 6. 복합 질문 테스트

여러 도구(think + kifrs_search + kifrs_term_definitions + kifrs_explore_related)를 조합하여 심층 분석하는 질문.

In [ ]:
result = run(
    agent,
    (
        "리스(lease)와 관련된 K-IFRS 기준서를 조사해주세요. "
        "먼저 '리스'의 정확한 용어정의를 확인하고, "
        "제1116호와 관련된 다른 기준서도 탐색해서 "
        "리스 회계처리의 핵심 원칙을 정리해주세요."
    ),
    thread_id="test-3",
)
print(result)

---
## 7. 멀티턴 대화 테스트

동일한 `thread_id`로 이전 맥락을 기반으로 후속 질문합니다.

In [ ]:
# 이전 test-3 대화를 이어서 후속 질문
result = run(
    agent,
    "방금 설명한 내용에서, 리스이용자의 사용권자산 측정 방법을 더 자세히 알려주세요.",
    thread_id="test-3",
)
print(result)

---
## 8. 스트리밍 테스트

응답을 실시간으로 확인합니다.

In [ ]:
for chunk in stream(agent, "금융자산의 분류 기준을 K-IFRS 제1109호에서 검색해서 요약해주세요.", thread_id="test-4"):
    print(chunk, end="", flush=True)
print()